# 56 — Supervisor's second review (October 2026): Holm register and threshold sensitivity

Run after notebooks 48, 50, 52, 51, 53, 54 and 55. The changes of the review that touch the main analysis (resampling counts set
to 1000, Holm columns next to every raw p, the rebuilt Figure 4.2) are in the modules those notebooks call. This notebook writes
the tables that only exist for the review. All the logic is in `viz/review2.py`.

| step | function | output (`LaTeX_v15/Data_for_writing/csv/`) |
|---|---|---|
| 2 Holm | `holm_register` | `holm_register.csv`: every test of the four families, raw p, Holm p, verdict at 0.05 before and after |
| 3a share rule | `fixed_share_tables`, `readable_architectures` | `q6_fixed_by_architecture[_detail]_share70.csv`, `_share90.csv` (80 % is the main table); at 20 designs: `…_min20_share70/80/90.csv`; `q6_fixed_by_architecture_readable.csv` |
| 3b rarefaction | `rarefied_hill_table`, `edge_cell` | `q7_hill_by_cohort_rarefied30.csv`, `…40.csv`, `…50.csv`; `q7_condition2_awt_2020_23_by_rarefaction.csv` |
| 4.2.3 | `q4.pairwise`, `funnel_steps`, `publication_lead` | `q4_coding_agreement.csv` (with Cohen's kappa), `q4_funnel_steps.csv`, `q4_publication_lead.csv`, `q4_publication_lead_summary.csv` |
| 3c acceptance | `acceptance_table` | `script_acceptance_thresholds.csv` (also `review/decisions/ACCURACY_THRESHOLD_SENSITIVITY.csv`) |

What changed and why is in `../RERUN_NOTES_2026-10.md` at the repository root.

In [1]:
import sys, pathlib
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR))
import pandas as pd
from IPython.display import display
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
from viz import config, thesis, review2
cfg = config.load()
ctx = thesis.context(cfg)

/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/labeling_evaluation/viz/frames.py:560: UserWarning: rapidfuzz not installed — fuzzy company matching disabled. Only exact lookup table matches will be applied.
  canon = g._normalise_company(sraw)


## Step 2. Holm within each family

Families: I.2 the directory readings against the dataset; II.1 the dimensions of Table 4.3; III.1 the architectures, one family per span; V the region × measure chi-squares, one family per assignee reading, and the regional trends, one family.

In [2]:
hr = review2.holm_register(ctx)
display(hr.groupby('family', sort=False).agg(tests=('p', 'size'), below_raw=('below 0.05, raw', 'sum'), below_holm=('below 0.05, Holm', 'sum')))
display(hr[hr['verdict changes']])

,tests,below_raw,below_holm
family,,,
"II.1 dimension × architecture (Cramér's V, permutation p)",10,10,10
"III.1 share × priority year, Spearman, 2008-2023",7,1,1
"III.1 share × priority year, Spearman, 2016-2023",7,0,0
I.2 dataset against the in-domain directory and its parts (chi-square),5,4,4
"V region × measure (chi-square), all filers",20,9,5
"V region × measure (chi-square), without Bell / Textron",20,7,0
V trend of a regional answer 2012-2023 (Spearman),42,8,3


,family,test,p,p (Holm),"below 0.05, raw","below 0.05, Holm",verdict changes
31,"V region × measure (chi-square), all filers",class mix (five classes + other) · 2016-19,0.0057,0.0798,True,False,True
36,"V region × measure (chi-square), all filers",propulsive-unit band · 2020-23,0.0036,0.0540,True,False,True
39,"V region × measure (chi-square), all filers",any tilting unit · 2016-19,0.0357,0.4284,True,False,True
45,"V region × measure (chi-square), all filers",family F · all years,0.0181,0.2353,True,False,True
51,"V region × measure (chi-square), without Bell / Textron",class mix (five classes + other) · 2016-19,0.0122,0.2440,True,False,True
52,"V region × measure (chi-square), without Bell / Textron",class mix (five classes + other) · 2020-23,0.0133,0.2527,True,False,True
56,"V region × measure (chi-square), without Bell / Textron",propulsive-unit band · 2020-23,0.0394,0.5910,True,False,True
57,"V region × measure (chi-square), without Bell / Textron",any tilting unit · all years,0.0232,0.4176,True,False,True
60,"V region × measure (chi-square), without Bell / Textron",any tilting unit · 2020-23,0.0296,0.5032,True,False,True
61,"V region × measure (chi-square), without Bell / Textron",Tilt Rotor · all years,0.0354,0.5664,True,False,True


## Step 3a. The 80 % share rule at 70 % and 90 %

Only the column of architectures where the leading option holds the share moves; V and p are the same shuffles.

In [3]:
from viz import thesis_v2_part2 as p2
base = p2._fixed_tables(ctx)[0]
tabs = review2.fixed_share_tables(ctx, (0.70, 0.90))
cmp = base[['dimension', 'architectures at 80 % or more']].copy()
for s, (t1, _) in tabs.items():
    cmp[f'at {round(100 * s)} %'] = t1[f'architectures at {round(100 * s)} % or more']
display(cmp)

,dimension,architectures at 80 % or more,at 70 %,at 90 %
0,propulsor units per design (band),0 of 7 read,0 of 7 read,0 of 7 read
1,tail type,0 of 7 read,1 of 7 read,0 of 7 read
2,places carrying propulsor units per design,1 of 7 read,1 of 7 read,1 of 7 read
3,booms present,2 of 9 read,3 of 9 read,2 of 9 read
4,propulsor enclosure (any ducted),1 of 7 read,2 of 7 read,0 of 7 read
5,fuselage shape,1 of 7 read,2 of 7 read,0 of 7 read
6,number of wings (winged designs),1 of 7 read,2 of 7 read,1 of 7 read
7,landing-gear type,0 of 7 read,3 of 7 read,0 of 7 read
8,fuselage motion hover to cruise,5 of 7 read,6 of 7 read,5 of 7 read
9,wing configuration,5 of 6 read,6 of 6 read,1 of 6 read


### 3a, second reading: an architecture read from 20 designs (2026-10-22)

The code reads an architecture from 10 answering designs (`thesis_q6.FIXED_MIN_DESIGNS`); the dissertation's 80 % table reads it from 20. The three shares again at 20, and which architectures each rule reads per dimension.

In [4]:
t20 = review2.fixed_share_tables(ctx, (0.70, 0.80, 0.90), min_designs=review2.MIN_READ_THESIS)
cmp20 = t20[0.80][0][['dimension']].copy()
for s, (t1, _) in t20.items():
    cmp20[f'at {round(100 * s)} %'] = t1[f'architectures at {round(100 * s)} % or more']
display(cmp20)
display(review2.readable_architectures(ctx))

,dimension,at 70 %,at 80 %,at 90 %
0,propulsor units per design (band),0 of 7 read,0 of 7 read,0 of 7 read
1,tail type,1 of 7 read,0 of 7 read,0 of 7 read
2,places carrying propulsor units per design,1 of 7 read,1 of 7 read,1 of 7 read
3,booms present,1 of 7 read,0 of 7 read,0 of 7 read
4,propulsor enclosure (any ducted),2 of 7 read,1 of 7 read,0 of 7 read
5,fuselage shape,2 of 7 read,1 of 7 read,0 of 7 read
6,number of wings (winged designs),1 of 6 read,0 of 6 read,0 of 6 read
7,landing-gear type,3 of 7 read,0 of 7 read,0 of 7 read
8,fuselage motion hover to cruise,6 of 7 read,5 of 7 read,5 of 7 read
9,wing configuration,6 of 6 read,5 of 6 read,1 of 6 read


,dimension,read from (designs),architectures read,which (answering designs)
0,propulsor units per design (band),10,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
1,propulsor units per design (band),20,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
2,tail type,10,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
3,tail type,20,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
4,places carrying propulsor units per design,10,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
5,places carrying propulsor units per design,20,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
6,booms present,10,9,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
7,booms present,20,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
8,propulsor enclosure (any ducted),10,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."
9,propulsor enclosure (any ducted),20,7,"Lift + Cruise (165), Tilt Propulsor (151), Combined Vectored Thrust (106), T..."


## Step 3b. Rarefied to 30, 40 (main) and 50 designs

The smallest complete cohort is 2012-15 with 58 designs, so every size draws without replacement from every complete cohort. The partial cohort 2024-26 is listed and never read.

In [5]:
h = review2.rarefied_hill_table(ctx)
read = h[~h['partial cohort (not read)']]
display(read.pivot_table(index=['level', 'cohort'], columns='rarefied to n', values=['²D', 'band low', 'below band (condition 2 met)'], aggfunc='first'))

band low               below band (condition 2 met)                   ²D              
rarefied to n       30     40     50                           30     40     50     30     40     50
level cohort                                                                                        
A     2012-15     4.04   4.12   4.16                        False  False  False   4.87   5.01   5.07
      2016-19     4.57   4.68   4.77                        False  False  False   5.75   5.96   6.09
      2020-23     4.64   4.73   4.81                         True   True   True   4.49   4.55   4.63
      <= 2011     4.10   4.18   4.22                        False  False  False   4.57   4.66   4.74
A·U   2012-15    11.06  11.89  12.36                         True   True   True   9.53  10.12  10.45
      2016-19    12.47  13.86  14.76                        False  False  False  14.53  16.21  17.57
      2020-23    12.58  13.85  14.74                        False  False  False  13.03  14.49  15.59
      <= 2011    11.01  12.15  12.68                         True   True   True   9.97  10.61  11.11
A·W·T 2012-15     6.75   6.97   7.17                        False  False  False   7.75   8.12   8.36
      2016-19     7.79   8.25   8.48                        False  False  False   9.92  10.58  11.13
      2020-23     7.91   8.30   8.63                         True  False   True   7.89   8.36   8.62
      <= 2011     6.88   7.03   7.25                        False  False  False   7.83   8.25   8.47
F     2012-15     2.96   3.03   3.08                        False  False  False   4.72   4.84   4.89
      2016-19     3.50   3.58   3.62                        False  False  False   4.73   4.84   4.93
      2020-23     3.55   3.62   3.63                         True   True   True   3.25   3.25   3.27
      <= 2011     3.04   3.11   3.08                        False  False  False   3.82   3.90   3.94

The one cell that changes with the size, A·W·T in 2020-23 (all designs): observed ²D, the middle 95 % of its rarefaction draws, the permutation band and the distance to its lower edge.

In [6]:
display(review2.edge_cell(ctx, h=h))

,cell,rarefied to n,designs,"²D (condition 2, 1000 permutations)",²D low,²D high,band low,band high,below band (condition 2 met),²D minus band low,as a share of the band width
0,"A·W·T, 2020-23, all designs",30,236,7.89,5.36,11.25,7.91,9.72,True,-0.02,-0.011
1,"A·W·T, 2020-23, all designs",40,236,8.36,6.06,11.27,8.30,10.37,False,0.06,0.029
2,"A·W·T, 2020-23, all designs",50,236,8.62,6.48,11.26,8.63,10.82,True,-0.01,-0.005


## Step 3c. Acceptance threshold of the script inputs

The rule in `review/HOW_THE_REVIEWS_WERE_DONE.md` accepts a source when the lower bound of its 95 % Wilson interval is at or above the threshold (default 90 %). Both readings, point share and lower bound, are shown at 60, 70, 80 and 90 %.

In [7]:
display(review2.acceptance_table())

,input,method,agree,of,share %,95 % low %,95 % high %,"accepted at 60%, share","accepted at 60%, lower bound","accepted at 70%, share","accepted at 70%, lower bound","accepted at 80%, share","accepted at 80%, lower bound","accepted at 90%, share","accepted at 90%, lower bound"
0,Development year,LLM (reading step),84,90,93.3,86.5,96.8,True,True,True,True,True,True,True,False
1,Development year,"script, exact year",49,81,60.5,50.0,70.1,True,False,False,False,False,False,False,False
2,Development year,"script, within 1 year",61,81,75.3,65.3,83.2,True,True,True,False,False,False,False,False
3,Development year,"script, same time window",58,81,71.6,61.4,80.0,True,True,True,False,False,False,False,False
4,TRL,LLM (reading step),50,50,100.0,96.1,100.0,True,True,True,True,True,True,True,True
5,TRL,"script, exact level",23,50,46.0,36.3,56.0,False,False,False,False,False,False,False,False
6,TRL,"script, same side of TRL 2",37,50,74.0,64.3,81.8,True,True,True,False,False,False,False,False
7,TRL,"script, within 1 level",37,50,74.0,64.3,81.8,True,True,True,False,False,False,False,False


## Section 4.2.3 (2026-10-22): agreement as Cohen's kappa, the funnel, the lead from the first publication

No new reading of any patent. Kappa: `viz/thesis_q4.py: pairwise` (the public aircraft's label rebuilt per pair from the existing match
record, `public_labels`; asserted to give back Table 2's agreement counts). Funnel and lead: `viz/review2.py`.

In [8]:
from viz import thesis_q4 as q4
display(q4.pairwise(ctx)[['comparison', 'agree', 'of', 'share', "Cohen's kappa"]])
display(review2.funnel_steps(ctx).set_index('population').T)
display(review2.publication_lead(ctx))
display(review2.publication_lead_summary(ctx))

,comparison,agree,of,share,Cohen's kappa
0,drawing label against the public aircraft,87,97,0.90,0.87
1,patent text against the public aircraft,89,97,0.92,0.90
2,drawing label against the patent text,93,97,0.96,0.95


population,all designs,"named designs (public counterpart, Table 2)",flown designs (TRL 5 or more),designs of the 12 index assignees that have flown,designs of the 14 index assignees
designs,627,97,68,42,47
designs with an architecture,625,97,68,42,47
largest architecture,Lift + Cruise,Lift + Cruise,Tilt Propulsor,Lift + Cruise,Lift + Cruise
its share,0.264,0.299,0.309,0.476,0.511
median propulsors per design,5.0,8.0,8.0,9.0,8.0
designs with a propulsor count,595,95,68,40,45
share with priority year 2020-2023,0.378,0.454,0.5,0.476,0.511
designs with a priority year,627,97,68,42,47
share Lift + Cruise,0.264,0.299,0.294,0.476,0.511
share Tilt Propulsor,0.242,0.278,0.309,0.19,0.191


,company,first priority,first publication,first flight (ARI),flown by the release,lead from first priority (years),lead from first publication (years)
0,Joby Aviation,2013,2015,2018.0,True,5.0,3.0
1,Guangzhou Ehang Intelligent Technology,2015,2017,2018.0,True,3.0,1.0
2,Embraer / Eve,2018,2019,2025.0,True,7.0,6.0
3,Honda,2018,2021,2024.0,True,6.0,3.0
4,AutoFlight,2019,2020,2022.0,True,3.0,2.0
5,Volocopter,2019,2020,2021.0,True,2.0,1.0
6,Wisk Aero,2020,2021,2025.0,True,5.0,4.0
7,Archer Aviation,2020,2021,2023.0,True,3.0,2.0
8,Beta Technologies,2021,2022,2020.0,True,-1.0,-2.0
9,Wollant Aviation,2022,2022,2023.0,True,1.0,1.0


,assignees,n,median lead from priority,range from priority,median lead from publication,range from publication,difference of the medians (publication - priority)
0,index assignees that have flown by the release,12,3.0,-1 to 7,1.5,-2 to 6,-1.5
1,every index assignee with a first-flight year (a planned year included),13,3.0,-1 to 7,2.0,-2 to 6,-1.0


## Write every table

In [9]:
for p in review2.write_all(cfg, ctx):
    print(p)

/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/LaTeX_v15/Data_for_writing/csv/q6_fixed_by_architecture_share70.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/LaTeX_v15/Data_for_writing/csv/q6_fixed_by_architecture_detail_share70.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/thesis_figures_v2/Part_II_design_space/II1_varying_attributes/appendix/q6_fixed_by_architecture_share70.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/1_labelling_analysis/labelling_analysis_thesis/thesis_figures_v2/Part_II_design_space/II1_varying_attributes/appendix/q6_fixed_by_architecture_detail_share70.csv
/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images